#Time Series - LAB 2

In [109]:
#import libraries
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

!pip install -U tsfresh
!pip install tslearn
!pip install pyts

#!pip install --upgrade threadpoolctl

import warnings
warnings.filterwarnings("ignore")

##Question 1: Classify the Italy Power Demand dataset using kNN and the tsfresh library.

In [ ]:
# Read the time series data - Train data
df = pd.read_csv("/content/ItalyPowerDemand_TRAIN.txt", header = None)

#Label
df_train_label = df.iloc[:, -1]
#Data
df = df.drop(df.columns[[-1]], axis=1)

#tsfresh needs all the time series to be "stacked up as a single time series" and separated by an id (therefore, we need to add this column).
df_train_data = pd.DataFrame()
for id in range(df.shape[0]):
  for i in range(df.shape[1]):
    data = [[id, i, df.iloc[id,i]]]
    data = pd.DataFrame(data)
    df_train_data = pd.concat([df_train_data, data], ignore_index=True)
    #df_train_data = df_train_data.append(data, ignore_index=True) #old version of pandas
df_train_data.columns = ['id', 'timestamp', 'value']

plt.figure(figsize=(8,4))
for xx in range(df.shape[0]):
    plt.plot(df.iloc[xx,:], "k-", alpha=.2)
plt.title('Training data')
plt.tight_layout()
plt.show()

df_train_data

In [ ]:
#https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html
import tsfresh
from tsfresh.feature_extraction import ComprehensiveFCParameters
fset = ComprehensiveFCParameters()

#manual selection of features
# fset = {'sum_values': None,
# 'median': None,
# 'mean': None,
# 'standard_deviation': None,
# 'variation_coefficient': None,
# 'variance': None,
# 'skewness': None,
# 'kurtosis': None,
# 'last_location_of_maximum': None,
# 'first_location_of_maximum': None,
# 'last_location_of_minimum': None,
# 'first_location_of_minimum': None,
# 'maximum': None,
# 'minimum': None}

#extract features
features_train=tsfresh.extract_features(df_train_data, column_id='id',column_sort='timestamp', column_value='value', default_fc_parameters=fset) # impute_function=impute)
print(f'Number of features: {features_train.shape[1]}')
features_train.describe()

In [ ]:
#There can be many non-values in extracted features that can be removed using the following lines of codes.
from tsfresh.utilities.dataframe_functions import impute
impute(features_train)

from tsfresh import select_features
features_train = select_features(features_train, df_train_label)
print(f'Number of selected features: {features_train.shape[1]}')

In [ ]:
# Read the time series data - Test data
df = pd.read_csv("/content/ItalyPowerDemand_TEST.txt", header = None)

#Label
df_test_label = df.iloc[:, -1]
#Data
df = df.drop(df.columns[[-1]], axis=1)

#tsfresh needs all the time-series to be "stacked up as a single time series" and separated by an id (therefore, we need to add this column).
df_test_data = pd.DataFrame()
for id in range(df.shape[0]):
  for i in range(df.shape[1]):
    data = [[id, i, df.iloc[id,i]]]
    data = pd.DataFrame(data)
    df_test_data = pd.concat([df_test_data, data], ignore_index=True)
    #df_test_data = df_test_data.append(data, ignore_index=True) #old version of pandas
df_test_data.columns = ['id', 'timestamp', 'value']

features_test=tsfresh.extract_features(df_test_data, column_id='id',column_sort='timestamp',column_value='value',default_fc_parameters = fset)
features_test = features_test[features_train.columns]

In [ ]:
#Classification using kNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

#Z-score
mean_features_train = features_train.mean()
std_features_train = features_train.std()

new_features_train = (features_train - mean_features_train)/std_features_train
new_features_test = (features_test - mean_features_train)/std_features_train

#features_train after Z-score: mean = 0 and std = 1
# print(new_features_train.std())
# print(new_features_train.mean())

#Calls the kNN classifier with k = 1
classifier_feature = KNeighborsClassifier(n_neighbors=1)
print('Before Z-score')
classifier_feature.fit(features_train, df_train_label)
print(classification_report(df_test_label, classifier_feature.predict(features_test)))

print('After Z-score')
classifier_feature.fit(new_features_train, df_train_label)
print(classification_report(df_test_label, classifier_feature.predict(new_features_test)))


##Question 2: Clustering the datasets using tslearn library

In [ ]:
from tslearn.clustering import TimeSeriesKMeans
from tslearn.datasets import CachedDatasets
from tslearn.datasets import UCR_UEA_datasets


print(CachedDatasets().list_datasets())
print(UCR_UEA_datasets().list_datasets())

# Load the data sets
X_train = CachedDatasets().load_dataset('Trace')[0]
#X_train = UCR_UEA_datasets().load_dataset('ItalyPowerDemand')[0]
#X_train = UCR_UEA_datasets().load_dataset('CBF')[0]
# Define parameters for each metric
euclidean_params = {'metric': 'euclidean'}
dba_params = {'metric': 'dtw'}
# Perform clustering for each metric
y_preds = []
aux = 0
for params in (euclidean_params, dba_params):
  km = TimeSeriesKMeans(n_clusters=3, random_state=0, **params)
  y_preds.append(km.fit_predict(X_train))

  plt.figure(figsize=(10,5))
  for yi in range(3):
    plt.subplot(1, 3, 1 + yi)
    for xx in X_train[y_preds[aux] == yi]:
        plt.plot(xx.ravel(), "k-", alpha=.2)
    plt.plot(km.cluster_centers_[yi].ravel(), "r-")
    plt.xlim(0, X_train.shape[1])
    plt.ylim(-4, 4)
    plt.title(f"{params} -> Cluster %d" % (yi + 1))

  plt.tight_layout()
  plt.show()
  aux += 1


In [ ]:
from tslearn.clustering import KShape
from tslearn.datasets import CachedDatasets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

seed = 0
np.random.seed(seed)
X_train, y_train, X_test, y_test = CachedDatasets().load_dataset("Trace")
# Keep first 3 classes and 50 first time series
X_train = X_train[y_train < 4]
X_train = X_train[:50]
np.random.shuffle(X_train)
# For this method to operate properly, prior scaling is required
X_train = TimeSeriesScalerMeanVariance().fit_transform(X_train)
sz = X_train.shape[1]

# kShape clustering
ks = KShape(n_clusters=3, verbose=True, random_state=seed)
y_pred = ks.fit_predict(X_train)

plt.figure()
for yi in range(3):
    plt.subplot(3, 1, 1 + yi)
    for xx in X_train[y_pred == yi]:
        plt.plot(xx.ravel(), "k-", alpha=.2)
    plt.plot(ks.cluster_centers_[yi].ravel(), "r-")
    plt.xlim(0, sz)
    plt.ylim(-4, 4)
    plt.title("Cluster %d" % (yi + 1))

plt.tight_layout()
plt.show()

##Question 3: Classify Cylinder-Bell-Funnel dataset using SAX-VSM using pyts library

In [ ]:
from pyts.datasets import make_cylinder_bell_funnel
#Cylinder-Bell-Funnel is a simulated data set with 930 instances (length = 128), 3 classes, 1 variable

X, y = make_cylinder_bell_funnel(n_samples=12, random_state=42)

plt.figure(figsize=(6, 8))
for i, classe in enumerate(['cylinder', 'bell', 'funnel']):
    plt.subplot(3, 1, i + 1)
    for x in X[y == i]:
        plt.plot(x, color='C0', linewidth=0.9)
    plt.title('Class: {}'.format(classe), fontsize=12)

plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
plt.show()

In [ ]:
from pyts.classification import SAXVSM
from sklearn.model_selection import train_test_split

X, y = make_cylinder_bell_funnel(n_samples=930, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=104, test_size=0.25, shuffle=True)


# SAXVSM transformation
saxvsm = SAXVSM(window_size=32, word_size=8, n_bins=4, strategy='uniform')
saxvsm.fit(X_train, y_train)
tfidf = saxvsm.tfidf_
vocabulary_length = len(saxvsm.vocabulary_)
X_new = saxvsm.decision_function(X_test)

# #Visualize the transformation
# plt.figure(figsize=(14, 5))
# width = 0.8/3

# plt.subplot(121)
# plt.bar(np.arange(vocabulary_length) - width, tfidf[0],
#         width=width, label='Class 1')
# plt.bar(np.arange(vocabulary_length), tfidf[1],
#         width=width, label='Class 2')
# plt.bar(np.arange(vocabulary_length) + width, tfidf[2],
#         width=width, label='Class 3')
# #plt.xticks(np.arange(vocabulary_length), np.vectorize(saxvsm.vocabulary_.get)(np.arange(vocabulary_length)), fontsize=14)
# plt.ylim((0, 17))
# plt.xlabel("Words", fontsize=14)
# plt.ylabel("tf-idf", fontsize=14)
# plt.title("tf-idf vector for each class (training set)", fontsize=15)
# plt.legend(loc='best')

accuracy = 0.0
for i in range(X_new.shape[0]):
  index = np.argmax(X_new[i])
  accuracy += index == y_test[i]

print(f'Accuracy: {100*accuracy/y_test.shape[0]}%')

##Question 4: Classify Basic Motions dataset using Recurrence Plot and CNN-2D

In [ ]:
#JointRecurrencePlot is an extension of a Recurrence Plot for multivariate time series.
from pyts.multivariate.image import JointRecurrencePlot
from pyts.datasets import load_basic_motions

#Basic Motions dataset is a multivariate time series with 80 instances (train = 40, test = 40), where each instance has 6 variables and 100 observations
#The data was generated as part of a student project where four students performed four activities whilst wearing a smart watch.
#The watch collects 3D accelerometer and a 3D gyroscope It consists of four classes, which are walking, resting, running and badminton.
X_train, X_test, y_train, y_test = load_basic_motions(return_X_y=True)

X_train = np.vstack([X_train, X_test])
y_train = np.hstack([y_train, y_test])

#num classes
num_classes = np.unique(y_train).shape[0]

# Recurrence plot transformation
jrp = JointRecurrencePlot(threshold='point', percentage=50)
X_rp_train = jrp.fit_transform(X_train)

# Show the results for the first time series
fig, axs = plt.subplots(2,2,figsize=(10,5))
aux = 0
for instance in range(2):
  for variable in range(X_train.shape[1]):
    axs[instance][aux].plot(np.arange(X_train[0].shape[1]), X_train[instance][variable])
  axs[instance][aux].set_title('Time Series', fontsize=12)
  aux += 1
  axs[instance][aux].imshow(X_rp_train[instance], cmap='binary', origin='lower')
  axs[instance][aux].set_title('Joint Recurrence Plot', fontsize=12)
  aux -= 1
plt.tight_layout()


In [ ]:
#It is needed to encode the classes as one-hot code
from sklearn.preprocessing import LabelEncoder

code = np.array(y_train)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(code)

In [ ]:
#Use CNN2D
from numpy import mean
from numpy import std
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import KFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.optimizers import SGD

In [ ]:
# scale pixels
def prep_pixels(train, test):
	# convert from integers to floats
	train_norm = train.astype('float32')
	test_norm = test.astype('float32')
	# normalize to range 0-1
	#train_norm = train_norm / 255.0
	#test_norm = test_norm / 255.0
	# return normalized images
	return train_norm, test_norm


# define cnn model
def define_model():
  model = Sequential()
  model.add(Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_uniform', input_shape=(100, 100, 1)))
  model.add(Conv2D(4, (3, 3), activation='relu', kernel_initializer='he_uniform'))
  model.add(MaxPooling2D((2, 2)))
  model.add(Flatten())
  model.add(Dropout(0.25))
  model.add(Dense(20, activation='relu', kernel_initializer='he_uniform'))
  model.add(Dense(num_classes, activation='softmax'))
	# compile model
  opt = SGD(learning_rate=0.001, momentum=0.9)
  model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
  return model


# evaluate a model using k-fold cross-validation
def evaluate_model(dataX, dataY, n_folds=2):
	scores, histories = list(), list()
	# prepare cross validation
	kfold = KFold(n_folds, shuffle=True, random_state=1)
	# enumerate splits
	for train_ix, test_ix in kfold.split(dataX,dataY):
		# define model
		model = define_model()
		# select data for train
		trainX, trainY, testX, testY = dataX[train_ix], dataY[train_ix], dataX[test_ix], dataY[test_ix]

		# select data for validation and test
		kfold_val_tes = KFold(2, shuffle=True, random_state=1)
		for val_ix, test_ix in kfold_val_tes.split(testX, testY):
			valX, valY, testX, testY = testX[val_ix], testY[val_ix], testX[test_ix], testY[test_ix]
			break

		# fit model
		history = model.fit(trainX, trainY, epochs=25, batch_size=8, validation_data=(valX, valY), verbose=1)
		# evaluate model
		_, acc = model.evaluate(testX, testY, verbose=0)
		print('> %.3f' % (acc * 100.0))
		# stores scores
		scores.append(acc)
		histories.append(history)
	return scores, histories

# plot diagnostic learning curves
def summarize_diagnostics(histories):
  plt.figure(figsize=(6, 8))
  for i in range(len(histories)):
	  # plot loss
	  plt.subplot(2, 1, 1)
	  plt.title('Cross Entropy Loss')
	  plt.plot(histories[i].history['loss'], color='blue', label='train')
	  plt.plot(histories[i].history['val_loss'], color='orange', label='test')
	  # plot accuracy
	  plt.subplot(2, 1, 2)
	  plt.title('Classification Accuracy')
	  plt.plot(histories[i].history['accuracy'], color='blue', label='train')
	  plt.plot(histories[i].history['val_accuracy'], color='orange', label='test')
  plt.show()

# summarize model performance
def summarize_performance(scores):
	# print summary
	print('Accuracy: mean=%.3f std=%.3f, n=%d' % (mean(scores)*100, std(scores)*100, len(scores)))
	# box and whisker plots of results
	plt.boxplot(scores)
	plt.show()

# run the test harness for evaluating a model
def run_test_harness():
	# load dataset

  trainX = X_rp_train
  trainY = to_categorical(y_train)

  # prepare pixel data
  trainX, trainX = prep_pixels(trainX, trainX)
  # evaluate model
  scores, histories = evaluate_model(trainX, trainY, n_folds=4) #k-fold cross-validation with k = 4
  # learning curves
  summarize_diagnostics(histories)
  # summarize estimated performance
  summarize_performance(scores)

# entry point, run the test harness
run_test_harness()

##Question 5: Classify Basic Motions using Deep Learning

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pyts.datasets import load_basic_motions

X_train, X_test, y_train, y_test = load_basic_motions(return_X_y=True)

#X_train = np.vstack([X_train, X_test])
#y_train = np.hstack([y_train, y_test])

num_classes = np.unique(y_train).shape[0]

#It is needed to encode the classes as one-hot code
from sklearn.preprocessing import LabelEncoder

code = np.array(y_train)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(code)
y_train = to_categorical(y_train)

code = np.array(y_test)
label_encoder = LabelEncoder()
y_test = label_encoder.fit_transform(code)
y_test = to_categorical(y_test)

X_train = X_train.reshape((X_train.shape[0]*100, 6))
X_test = X_test.reshape((X_test.shape[0]*100, 6))

scaler_type=StandardScaler()
X_train = scaler_type.fit_transform(X_train)
X_test = scaler_type.fit_transform(X_test)

X_train = X_train.reshape((40, 100, 6))
X_test = X_test.reshape((40, 100, 6))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pyts.datasets import load_basic_motions

X_train, X_test, y_train, y_test = load_basic_motions(return_X_y=True)

X_data = np.vstack([X_train, X_test])
y_data = np.hstack([y_train, y_test]) #labels

num_classes = np.unique(y_data).shape[0]

#It is needed to encode the classes as one-hot code
from sklearn.preprocessing import LabelEncoder

code = np.array(y_data)
label_encoder = LabelEncoder()
y_data = label_encoder.fit_transform(code)
y_data = to_categorical(y_data)


kfold = KFold(2, shuffle=True, random_state=1)
# enumerate splits
for train_ix, test_ix in kfold.split(X_data,y_data):
  # define model
  # select rows for train
  X_train, y_train, X_test, y_test = X_data[train_ix], y_data[train_ix], X_data[test_ix], y_data[test_ix]

  kfold_val_tes = KFold(2, shuffle=True, random_state=1)
  for val_ix, test_ix in kfold_val_tes.split(X_test, y_test):
    X_val, y_val, X_test, y_test = X_test[val_ix], y_test[val_ix], X_test[test_ix], y_test[test_ix]
    break
  break

X_train = X_train.reshape((X_train.shape[0]*100, 6))
X_val = X_val.reshape((X_val.shape[0]*100, 6))
X_test = X_test.reshape((X_test.shape[0]*100, 6))


scaler_type = StandardScaler()
scaler_type.fit(X_train)

print(scaler_type.mean_)
print(scaler_type.var_)

#Standardize features by removing the mean and scaling to unit variance (using mean and variance of data train).
X_train = scaler_type.transform(X_train)
X_val = scaler_type.transform(X_val)
X_test = scaler_type.transform(X_test)

#X_train = scaler_type.fit_transform(X_train)
#X_val = scaler_type.fit_transform(X_val)
#X_test = scaler_type.fit_transform(X_test)

X_train = X_train.reshape((int(X_train.shape[0]/100), 100, 6))
X_val = X_val.reshape((int(X_val.shape[0]/100), 100, 6))
X_test = X_test.reshape((int(X_test.shape[0]/100), 100, 6))


# Show the time series after applying StandardScaler
fig, axs = plt.subplots(2,1,figsize=(10,5))
aux = 0
for instance in range(2):
  for variable in range(X_train.shape[2]):
    axs[instance].plot(np.arange(X_train[0].shape[0]), X_train[instance,:,variable])
  axs[instance].set_title('Time Series', fontsize=12)
plt.tight_layout()

In [ ]:
from keras import metrics
import keras
import tensorflow as tf
#import os

epochs = 50
batch_size = 8
window_length = 10
train_data_size = 0.75

#Early stopping is used to avoid overfitting
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', min_delta=1e-2, patience=5, verbose=0, mode='auto',
    baseline=None, restore_best_weights=True)


train_x = X_train
train_y = y_train
valid_x = X_val
valid_y = y_val
test_x = X_test
test_y = y_test

# define cnn model
def define_model():
  model = Sequential()
  model.add(keras.layers.Conv1D(16, (3), activation='relu', kernel_initializer='he_uniform', input_shape=(100, 6)))
  model.add(keras.layers.Conv1D(4, (3), activation='relu', kernel_initializer='he_uniform'))
  model.add(keras.layers.MaxPooling1D((2)))
  model.add(keras.layers.LSTM(4, kernel_initializer='he_uniform', return_sequences=True))
  model.add(Flatten())
  model.add(Dropout(0.25))
  model.add(Dense(40, activation='relu', kernel_initializer='he_uniform'))
  model.add(Dense(num_classes, activation='softmax'))
	# compile model
  opt = tf.keras.optimizers.Adam(learning_rate=0.001)
  model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
  model.build()
  print(model.summary())
  return model


#Train network
model = define_model()
model_history = model.fit(x=train_x, y=train_y, validation_data=(valid_x,valid_y), epochs=epochs, batch_size=batch_size, shuffle=True, callbacks=[early_stop], verbose=1)

In [ ]:
#plot Train loss and Validation loss
fig, (ax1) = plt.subplots(1, 1, sharey=True,figsize=(15,7))

ax1.plot(model_history.history['loss'], label='Train loss')
ax1.plot(model_history.history['val_loss'], label='Validation loss')
ax1.legend(loc='best')
ax1.set_title('Deep Learning')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss Function')

plt.show()

In [ ]:
# evaluate model
_, acc = model.evaluate(test_x, test_y, verbose=0)
print('> %.3f' % (acc * 100.0))

##Question 6: Classify segments of Well-01.csv using Deep Learning. Use SIM-01.csv, SIM-02.csv, and SIM-03.csv for training. Each segment must have length = 100.

In [ ]:
df = pd.read_csv("/content/SIM-01.csv", header = 0)
#Label
df_train_label = df.iloc[:, -1]
#Data
X_train = df.drop(df.columns[[0, -1]], axis=1) #remove timestamp and label

df = pd.read_csv("/content/SIM-02.csv", header = 0)
#Label
df_val_label = df.iloc[:, -1]
#Data
X_val = df.drop(df.columns[[0, -1]], axis=1) #remove timestamp and label

df = pd.read_csv("/content/WELL-01.csv", header = 0)
#Label
df_test_label = df.iloc[:, -1]
#Data
X_test = df.drop(df.columns[[0, -1]], axis=1) #remove timestamp and label

#Standardize features by removing the mean and scaling to unit variance.
scaler_type = StandardScaler()
scaler_type.fit(X_train)
X_train = scaler_type.transform(X_train)
X_val = scaler_type.transform(X_val)
X_test = scaler_type.transform(X_test)

#train data
train_X = []
train_y = []
for i in range(0,X_train.shape[0],100):
    train_X.append(X_train[i:i+100,:])
    train_y.append(df_train_label[i+100-1])

#val data
val_X = []
val_y = []
for i in range(0,X_val.shape[0],100):
    val_X.append(X_val[i:i+100,:])
    val_y.append(df_val_label[i+100-1])

#test data
test_X = []
test_y = []
for i in range(0,X_test.shape[0],100):
    test_X.append(X_test[i:i+100,:])
    test_y.append(df_test_label[i+100-1])


X_train = np.array(train_X)
y_train = np.array(train_y)

X_val = np.array(val_X)
y_val = np.array(val_y)

X_test = np.array(test_X)
y_test = np.array(test_y)

code = np.array(y_train)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(code)
y_train = to_categorical(y_train)

code = np.array(y_val)
label_encoder = LabelEncoder()
y_val = label_encoder.fit_transform(code)
y_val = to_categorical(y_val)

code = np.array(y_test)
label_encoder = LabelEncoder()
y_test = label_encoder.fit_transform(code)
y_test = to_categorical(y_test)

num_classes = np.unique(y_train).shape[0]